Testing the image and annotation loading and seeing the returned results

In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class EmotionDatasetRAM(Dataset):
    def __init__(self, img_dir="images", ann_dir="annotations", transform=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transform = transform

        # Collect IDs
        self.ids = [os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith((".jpg", ".png"))]
        self.ids.sort()

        # Preload everything into memory
        self.data = []
        for _id in self.ids:
            # Image
            img_path = os.path.join(img_dir, f"{_id}.jpg")
            if not os.path.exists(img_path):
                img_path = os.path.join(img_dir, f"{_id}.png")
            image = Image.open(img_path).convert("RGB")

            # Annotations
            exp = int(np.load(os.path.join(ann_dir, f"{_id}_exp.npy")))
            val = float(np.load(os.path.join(ann_dir, f"{_id}_val.npy")))
            aro = float(np.load(os.path.join(ann_dir, f"{_id}_aro.npy")))
            lnd = np.load(os.path.join(ann_dir, f"{_id}_lnd.npy")).astype(np.float32).reshape(-1)  # (136,)

            self.data.append({
                "id": _id,
                "image": image,
                "expression": exp,
                "valence": val,
                "arousal": aro,
                "landmarks": lnd
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        img = sample["image"]
        if self.transform:
            img = self.transform(img)

        return {
            "image": img,
            "landmarks": torch.tensor(sample["landmarks"], dtype=torch.float32),
            "expression": torch.tensor(sample["expression"], dtype=torch.long),
            "valence": torch.tensor(sample["valence"], dtype=torch.float32),
            "arousal": torch.tensor(sample["arousal"], dtype=torch.float32),
        }


The actual model architecture

In [ ]:
import torch.nn as nn

class EmotionNet(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        # CNN for images
        #a bunch of conv layers to increase the size and relu to flesh them out and a max pool to reduce the size
        self.image_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.img_fc = nn.Linear(256, 256)

        # MLP for landmarks
        #simple ann layer to reduce the size of the landmarks
        self.lnd_encoder = nn.Sequential(
            nn.Linear(136, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )

        # Fusion
        #combine the two features and pass them through a relu and dropout layer
        self.fusion_fc = nn.Sequential(
            nn.Linear(256+64, 256), nn.ReLU(), nn.Dropout(0.3)
        )

        # Heads
        #the three annotations we need to predict
        self.classifier = nn.Linear(256, num_classes)   # expression
        self.valence_head = nn.Linear(256, 1)          # regression
        self.arousal_head = nn.Linear(256, 1)          # regression

    def forward(self, image, landmarks):
        img_feat = self.image_encoder(image)
        img_feat = img_feat.view(img_feat.size(0), -1)
        img_feat = self.img_fc(img_feat)

        lnd_feat = self.lnd_encoder(landmarks)

        fused = torch.cat([img_feat, lnd_feat], dim=1)
        h = self.fusion_fc(fused)

        return {
            "logits": self.classifier(h),
            "valence": self.valence_head(h).squeeze(1),
            "arousal": self.arousal_head(h).squeeze(1)
        }


Normalizing the images and loading them to the model + the loss and optimizer

In [3]:
import torch.optim as optim
from torch.utils.data import random_split
torch.cuda.empty_cache() #clearing the cache otherwiese it goes out of memory
# transforms for images
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# dataset & loader
ds = EmotionDatasetRAM("images", "annotations", transform=transform)
loader = DataLoader(ds, batch_size=64, shuffle=True)

# model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionNet().to(device)

# losses
cls_loss_fn = nn.CrossEntropyLoss()
reg_loss_fn = nn.MSELoss()

# optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# training loop
for epoch in range(1, 101):  # 100 epochs
    model.train()
    running_loss = 0.0

    for batch in loader:
        imgs = batch["image"].to(device)
        lnds = batch["landmarks"].to(device)
        expr = batch["expression"].to(device)
        vals = batch["valence"].to(device)
        aros = batch["arousal"].to(device)

        out = model(imgs, lnds)

        # losses
        loss_cls = cls_loss_fn(out["logits"], expr)
        loss_val = reg_loss_fn(out["valence"], vals)
        loss_aro = reg_loss_fn(out["arousal"], aros)

        loss = loss_cls + loss_val + loss_aro

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(ds)
    print(f"Epoch [{epoch}/100] Loss: {epoch_loss:.4f}")


Epoch [1/100] Loss: 4.1172
Epoch [2/100] Loss: 2.4633
Epoch [3/100] Loss: 2.4432
Epoch [4/100] Loss: 2.4359
Epoch [5/100] Loss: 2.4315
Epoch [6/100] Loss: 2.4249
Epoch [7/100] Loss: 2.4180
Epoch [8/100] Loss: 2.4167
Epoch [9/100] Loss: 2.4137
Epoch [10/100] Loss: 2.4047
Epoch [11/100] Loss: 2.4028
Epoch [12/100] Loss: 2.4020
Epoch [13/100] Loss: 2.3946
Epoch [14/100] Loss: 2.3917
Epoch [15/100] Loss: 2.3834
Epoch [16/100] Loss: 2.3749
Epoch [17/100] Loss: 2.3682
Epoch [18/100] Loss: 2.3642
Epoch [19/100] Loss: 2.3671
Epoch [20/100] Loss: 2.3537
Epoch [21/100] Loss: 2.3530
Epoch [22/100] Loss: 2.3566
Epoch [23/100] Loss: 2.3398
Epoch [24/100] Loss: 2.3338
Epoch [25/100] Loss: 2.3357
Epoch [26/100] Loss: 2.3308
Epoch [27/100] Loss: 2.3256
Epoch [28/100] Loss: 2.3205
Epoch [29/100] Loss: 2.3354
Epoch [30/100] Loss: 2.3018
Epoch [31/100] Loss: 2.2965
Epoch [32/100] Loss: 2.2883
Epoch [33/100] Loss: 2.2887
Epoch [34/100] Loss: 2.3201
Epoch [35/100] Loss: 2.2968
Epoch [36/100] Loss: 2.2997
E

Complete code without any breaks in the cells running on 20 cells

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision import transforms

# ============================================================
# Dataset (Images + Annotations in RAM)
# ============================================================
class EmotionDatasetRAM(Dataset):
    def __init__(self, img_dir="images", ann_dir="annotations", transform=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transform = transform

        # collect IDs from image filenames
        self.ids = [os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith((".jpg", ".png"))]
        self.ids.sort()

        # preload into memory
        self.data = []
        for _id in self.ids:
            # image
            img_path = os.path.join(img_dir, f"{_id}.jpg")
            if not os.path.exists(img_path):
                img_path = os.path.join(img_dir, f"{_id}.png")
            image = Image.open(img_path).convert("RGB")

            # annotations
            exp = int(np.load(os.path.join(ann_dir, f"{_id}_exp.npy")))
            val = float(np.load(os.path.join(ann_dir, f"{_id}_val.npy")))
            aro = float(np.load(os.path.join(ann_dir, f"{_id}_aro.npy")))
            lnd = np.load(os.path.join(ann_dir, f"{_id}_lnd.npy")).astype(np.float32).reshape(-1)

            self.data.append({
                "image": image,
                "expression": exp,
                "valence": val,
                "arousal": aro,
                "landmarks": lnd
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        img = sample["image"]
        if self.transform:
            img = self.transform(img)

        return {
            "image": img,
            "landmarks": torch.tensor(sample["landmarks"], dtype=torch.float32),
            "expression": torch.tensor(sample["expression"], dtype=torch.long),
            "valence": torch.tensor(sample["valence"], dtype=torch.float32),
            "arousal": torch.tensor(sample["arousal"], dtype=torch.float32),
        }

# ============================================================
# Model (CNN for Images + MLP for Landmarks + Fusion)
# ============================================================
from torchvision.models import resnet18

class EmotionNet(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        # Use pretrained ResNet18
        base = resnet18(pretrained=True)
        base.fc = nn.Identity()  # remove final layer
        self.image_encoder = base
        self.img_fc = nn.Linear(512, 256)  # resnet18 outputs 512-dim features

        # landmarks branch
        self.lnd_encoder = nn.Sequential(
            nn.Linear(136, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )

        # fusion
        self.fusion_fc = nn.Sequential(
            nn.Linear(256+64, 256), nn.ReLU(), nn.Dropout(0.2)
        )

        # heads
        self.classifier = nn.Linear(256, num_classes)
        self.valence_head = nn.Linear(256, 1)
        self.arousal_head = nn.Linear(256, 1)

    def forward(self, image, landmarks):
        img_feat = self.image_encoder(image)
        img_feat = self.img_fc(img_feat)

        lnd_feat = self.lnd_encoder(landmarks)

        fused = torch.cat([img_feat, lnd_feat], dim=1)
        h = self.fusion_fc(fused)

        return {
            "logits": self.classifier(h),
            "valence": self.valence_head(h).squeeze(1),
            "arousal": self.arousal_head(h).squeeze(1)
        }


# ============================================================
# Training + Validation
# ============================================================
def accuracy(logits, targets):
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()

def train_model(img_dir="images", ann_dir="annotations", epochs=20, batch_size=32, lr=1e-4):
    # transforms
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],
                             std=[0.229,0.224,0.225])
    ])

    # dataset
    ds = EmotionDatasetRAM(img_dir, ann_dir, transform=transform)

    # train/val split
    train_size = int(0.8 * len(ds))
    val_size = len(ds) - train_size
    train_ds, val_ds = random_split(ds, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    # model, loss, optimizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EmotionNet().to(device)

    cls_loss_fn = nn.CrossEntropyLoss()
    reg_loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # training loop
    for epoch in range(1, epochs+1):
        # TRAIN
        model.train()
        train_loss, train_acc = 0.0, 0.0
        for batch in train_loader:
            imgs = batch["image"].to(device)
            lnds = batch["landmarks"].to(device)
            expr = batch["expression"].to(device)
            vals = batch["valence"].to(device)
            aros = batch["arousal"].to(device)

            out = model(imgs, lnds)

            # losses
            loss_cls = cls_loss_fn(out["logits"], expr)
            loss_val = reg_loss_fn(out["valence"], vals)
            loss_aro = reg_loss_fn(out["arousal"], aros)
            loss = loss_cls + loss_val + loss_aro

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * imgs.size(0)
            train_acc  += accuracy(out["logits"], expr) * imgs.size(0)

        train_loss /= len(train_ds)
        train_acc  /= len(train_ds)

        # VALIDATION
        model.eval()
        val_loss, val_acc = 0.0, 0.0
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch["image"].to(device)
                lnds = batch["landmarks"].to(device)
                expr = batch["expression"].to(device)
                vals = batch["valence"].to(device)
                aros = batch["arousal"].to(device)

                out = model(imgs, lnds)

                loss_cls = cls_loss_fn(out["logits"], expr)
                loss_val = reg_loss_fn(out["valence"], vals)
                loss_aro = reg_loss_fn(out["arousal"], aros)
                loss = loss_cls + loss_val + loss_aro

                val_loss += loss.item() * imgs.size(0)
                val_acc  += accuracy(out["logits"], expr) * imgs.size(0)

        val_loss /= len(val_ds)
        val_acc  /= len(val_ds)

        print(f"Epoch [{epoch}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    return model

# ============================================================
# Run Training
# ============================================================
if __name__ == "__main__":
    model = train_model(img_dir="images", ann_dir="annotations", epochs=20, batch_size=32, lr=1e-4)


/home/kintopi/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kintopi/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/20] Train Loss: 4.1043 | Train Acc: 0.1644 Val Loss: 2.1987 | Val Acc: 0.3113
Epoch [2/20] Train Loss: 2.0442 | Train Acc: 0.3545 Val Loss: 2.0047 | Val Acc: 0.3800
Epoch [3/20] Train Loss: 1.5606 | Train Acc: 0.5570 Val Loss: 1.8771 | Val Acc: 0.4188
Epoch [4/20] Train Loss: 0.9855 | Train Acc: 0.7781 Val Loss: 2.0063 | Val Acc: 0.4125
Epoch [5/20] Train Loss: 0.5647 | Train Acc: 0.9206 Val Loss: 2.0277 | Val Acc: 0.4088
Epoch [6/20] Train Loss: 0.3269 | Train Acc: 0.9806 Val Loss: 2.1012 | Val Acc: 0.4037
Epoch [7/20] Train Loss: 0.2310 | Train Acc: 0.9916 Val Loss: 2.0680 | Val Acc: 0.4425
Epoch [8/20] Train Loss: 0.1918 | Train Acc: 0.9959 Val Loss: 2.0771 | Val Acc: 0.4338
Epoch [9/20] Train Loss: 0.1612 | Train Acc: 0.9975 Val Loss: 2.0471 | Val Acc: 0.4238
Epoch [10/20] Train Loss: 0.1417 | Train Acc: 0.9991 Val Loss: 2.0329 | Val Acc: 0.4500
Epoch [11/20] Train Loss: 0.1138 | Train Acc: 0.9991 Val Loss: 2.0740 | Val Acc: 0.4263
Epoch [12/20] Train Loss: 0.1095 | Train 